# Chapter 49 — FAISS Fundamentals: High-Speed Vector Similarity Search

Faiss (Facebook AI Similarity Search) is an open-source C++ library with Python and CUDA/GPU bindings developed by Meta's Fundamental AI Research (FAIR) group for efficient similarity search and clustering of dense vectors.

# Install dependencies

In [2]:
%pip install -q faiss-cpu sentence-transformers numpy


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# 2. Import libraries

In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# 3. Create Documents and metadata

In [5]:
documents = [
    "Banks monitor cross-border payments to identify suspicious transaction patterns.",
    "Money laundering conceals the illegal origin of funds.",
    "Customer due diligence verifies customer identity and assesses customer risk.",
    "Credit risk models estimate the probability that a borrower will default.",
    "Machine learning models detect unusual patterns in financial transactions.",
    "Fraud detection systems identify payments that may be unauthorized or deceptive.",
    "Investigators review unusual international wire transfers for signs of financial crime.",
    "Know Your Customer checks help banks understand who their customers are.",
    "Sanctions screening checks whether a person or organization appears on a sanctions list.",
    "A neural network can classify images and process natural language.",
    "Transaction monitoring rules flag payments that match defined risk scenarios.",
    "Rapid movement of funds between multiple accounts may require investigation."
]

ids = [f"doc_{i:02d}" for i in range(1, len(documents) + 1)]
metadatas = [
    {"category": "AML", "source": "monitoring"},
    {"category": "AML", "source": "financial_crime"},
    {"category": "KYC", "source": "customer_risk"},
    {"category": "Credit Risk", "source": "risk_model"},
    {"category": "AML", "source": "machine_learning"},
    {"category": "Fraud", "source": "fraud_detection"},
    {"category": "AML", "source": "investigation"},
    {"category": "KYC", "source": "customer_risk"},
    {"category": "Sanctions", "source": "screening"},
    {"category": "AI", "source": "general_ai"},
    {"category": "AML", "source": "monitoring"},
    {"category": "AML", "source": "investigation"}
]


In [6]:
assert len(documents) == len(ids) == len(metadatas)


In [7]:
print("Documents:", len(documents))
print("IDs:", len(ids))
print("Metadata records:", len(metadatas))

Documents: 12
IDs: 12
Metadata records: 12


# 4. Create Embeddings

In [31]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
document_embeddings = model.encode(
    documents)
document_embeddings = np.asarray(document_embeddings, dtype="float32")


In [32]:
document_embeddings

array([[ 0.04110211, -0.03193491, -0.0718735 , ..., -0.04050156,
         0.07513548, -0.04125776],
       [-0.02550224,  0.01433703, -0.04527479, ..., -0.02897048,
         0.15225163, -0.02563388],
       [-0.08785745,  0.03319031,  0.04560452, ...,  0.03959312,
         0.05443267, -0.10720216],
       ...,
       [-0.03690125, -0.09516034,  0.0357466 , ...,  0.13678594,
         0.04741392, -0.05872878],
       [ 0.01484282,  0.02608941, -0.12955011, ..., -0.02620672,
         0.03389435, -0.07329618],
       [ 0.03782288, -0.04427537, -0.05512969, ..., -0.02668406,
         0.04062993, -0.06660976]], dtype=float32)

In [33]:
document_embeddings.shape

(12, 384)

In [34]:
document_embeddings.dtype

dtype('float32')

In [35]:
np.linalg.norm(document_embeddings[0])

np.float32(0.99999994)

# 5. Build a FAISS index
IndexFlatIP is a flat index using inner product. It compares a query against the stored vectors directly, giving exact results for this metric. It is useful for learning and small collections; very large collections may require other index types to trade off speed, memory, and accuracy.

In [36]:
dimension = document_embeddings.shape[1]

In [37]:
dimension

384

In [38]:
index = faiss.IndexFlatIP(dimension)

In [39]:
index.add(document_embeddings)

In [40]:
# Vector Dimension expected 
index.d

384

In [41]:
# Vectors indexed 

index.ntotal

12

# 6. Run your first semantic search
FAISS returns similarity scores and zero-based vector positions. These positions are not string IDs; we use them to look up the original text and metadata.

In [46]:
query = "How can a bank detect suspicious international payments?"

query_embedding = model.encode(query, convert_to_numpy=True)

query_embedding = np.asanyarray(query_embedding, dtype="float32")



In [47]:
query_embedding.shape

(384,)

In [53]:
query_embedding = np.asarray(query_embedding, dtype="float32")

# adding one dimension 
query_embedding = query_embedding.reshape(1, -1)

k = 3
scores, positions = index.search(query_embedding, k)



In [54]:
scores

array([[0.7641874 , 0.65887845, 0.57527256]], dtype=float32)

In [55]:
positions

array([[0, 6, 5]])

In [71]:
for rank, (score, pos) in enumerate(zip(scores.flatten(), positions.flatten()), start=1):
    if pos == -1:
        continue  # Skip if FAISS didn't find enough matches
        
    print(f"\nRank: {rank}")
    print(f"Index Position: {pos}")
    print(f"Distance Score: {score:.4f}")
    print(f"Text Content: {documents[pos]}")


Rank: 1
Index Position: 5
Distance Score: 0.7642
Text Content: Fraud detection systems identify payments that may be unauthorized or deceptive.
